# **Planning**

- When students are sorted purely by school, gender, and then CGPA:
  - The top of the list ends up filled with high-CGPA students from similar schools or the same gender.
  - The bottom is filled with lower-CGPA students, creating uneven skill distribution.
  - This simple sorting approach doesn’t achieve the goal of *mixing* strong and weak students across teams, and also makes the grouping more specific than diversed.

---

# **Solving the Issue**

To create balanced and diverse teams, a combined performance indicator called **t-score** is used.  
It reflects both a student’s academic standing (CGPA) and diversity factors like school and gender representation.

### **Step 1: Calculate the t-score**

1. Identify the **lowest** and **highest** CGPA within each tutorial group.  
2. Compute the **CGPA range**:  
   `highest CGPA − lowest CGPA`  
3. For each student, calculate:  
`t_score = (student CGPA − lowest CGPA) / (CGPA range)`

- Add a school diversity bonus that increases for students from less common schools in the tutorial group. The rarer the school, the higher the bonus, up to a maximum of **+0.1**.  
- Add **+0.2** if the student is from the least-represented gender.

This ensures that underrepresented groups receive a small bonus, encouraging more balanced diversity.

---

### **Step 2: Arrange Students Using Alternating High–Low Ordering**

Once every student has a t-score, they are arranged in an **alternating high-low order**:
`[highest, lowest, 2nd highest, 2nd lowest, …]`


This ordering distributes high and low performing students evenly before team assignment,  
preventing early clustering of either extreme.

---

### **Step 3: Assign Students to Teams**

Students are then grouped into teams by evaluating how well each student fits a team’s current balance profile.  
Each possible placement receives a **fit score**, the lower the score, the more balanced the result.

This process ensures each team:
- Has a mix of strong and weak academic performers.  
- Maintains diversity across gender and school attributes.

---

## **Repair Teams**

After initial grouping, some teams may still deviate from the overall tutorial mean t-score (e.g., ±0.1).
This means certain teams are slightly stronger or weaker than others, so a rebalancing phase is applied.

### **Goal**

To further minimize imbalance between teams through **repeated 1-for-1 swaps** between the most unbalanced ones,  
without violating any diversity or size constraints.

### **How It Works**

1. **Identify Outliers**  
   - Compute each team’s mean t-score and deviation from the tutorial mean.  
   - Group teams into “high” (above `+threshold`) and “low” (below `−threshold`) categories.

2. **Search for the Best Swap**  
   - For every high–low team pair, test all possible student swaps.  
   - Skip swaps that would exceed **school** or **gender caps**.  
   - For each valid swap, calculate how much it reduces the total deviation from the tutorial mean.  
   - Keep the swap that offers the **largest improvement**.

3. **Commit and Repeat**  
   - Apply the best swap found in the current iteration.  
   - Update the new team averages (`team_tscore`).  
   - Repeat until no better swaps remain or the iteration limit is reached.

### **Summary**

- Performs **repeated 1-for-1 swaps** to reduce imbalance.  
- Preserves **team size**, **school caps**, and **gender caps**.  
- Stops automatically when no better swaps remain or a safety limit is reached.  
- Gradually pulls team averages closer to the tutorial mean without major reshuffling.

---

## **Why Alternating High–Low Ordering Is Essential**

1. **If sorted by CGPA (high → low):**
   - Early teams receive all top students.  
   - Later teams receive mostly low scorers.  
   - → *Results in unbalanced teams and wide CGPA gaps.*

2. **If arranged randomly:**
   - Some teams may still end up uneven by chance.  
   - → Requires more adjustments later to fix imbalances.

3. **If arranged in alternating high–low order (recommended method):**
   - High and low scorers are distributed evenly from the start.  
   - Teams begin closer to the overall mean performance.  
   - → Allows the balancing step to focus on fine-tuning school and gender diversity instead of correcting major academic gaps.


#Get Team Size and import libraries



In [ ]:
# import libs
import csv
import io
import os
import requests

# team size config: allows 4–10 with safe defaults
MIN_TEAM_SIZE = 4
MAX_TEAM_SIZE = 10
DEFAULT_TEAM_SIZE = 5

# check if input is integer
def _parse_int(s):
    try:
        return int(s)
    except Exception:
        return None

# ask user for team size
def get_team_size():
    try:
        raw = input(
            f"Enter team size ({MIN_TEAM_SIZE}-{MAX_TEAM_SIZE}) [{DEFAULT_TEAM_SIZE}]: "
        ).strip()
        ts = _parse_int(raw) if raw else DEFAULT_TEAM_SIZE
        if MIN_TEAM_SIZE <= ts <= MAX_TEAM_SIZE:
            return ts
        else:
            print(f"Invalid team size; using default {DEFAULT_TEAM_SIZE}.")
    except Exception:
        pass

    return DEFAULT_TEAM_SIZE

TEAM_SIZE = get_team_size()
print(f"Using team size: {TEAM_SIZE}")

# Import records.csv and map to student_list array


In [ ]:
# read raw data from Github

# GITHUB_CSV_URL = "https://raw.githubusercontent.com/lekekbot/ntu-cs/main/SC1003/group%20project/records.csv"

student_list = []

# resp = requests.get(GITHUB_CSV_URL, timeout=30)
# resp.raise_for_status()  # will error clearly if the URL is bad

# csv_stream = io.StringIO(resp.content.decode("utf-8-sig"))

# reader = csv.DictReader(csv_stream)

# for row in reader:
#     # convert CGPA from string to float
#     cgpa_str = (row.get("CGPA") or "").strip()
#     try:
#         cgpa_val = float(cgpa_str) if cgpa_str else None
#     except ValueError:
#         cgpa_val = None

#     student = {
#         "tutorial":   (row.get("Tutorial Group") or "").strip(),
#         "student_id": (row.get("Student ID") or "").strip(),
#         "name":       (row.get("Name") or "").strip(),
#         "school":     (row.get("School") or "").strip(),
#         "gender":     (row.get("Gender") or "").strip(),
#         "cgpa":       cgpa_val,
#     }
#     student_list.append(student)

# !change back to original csv file retrieval before submission

with open("records.csv", newline="", encoding="utf-8") as file:
      reader = csv.DictReader(file)

      # Process each student row
      for row in reader:
          student = {
              "tutorial": row["Tutorial Group"].strip(),
              "student_id": row["Student ID"].strip(),
              "name": row["Name"].strip(),
              "school": row["School"].strip(),
              "gender": row["Gender"].strip(),
              "cgpa": float(row["CGPA"]),
          }
          student_list.append(student)


# quick sanity check to ensure that records are retrieved properly
print(f"Loaded {len(student_list)} students. First record:")
print(student_list[0] if student_list else "No rows found.")

# Loop into student_list to sort them based on their tutorial group
The sorted group will be a dictionary of array
```
{
  "G-1": [
    {
      'tutorial': 'G-1',
      'student_id': '5002',
      'name': 'Aarav Singh',
      'school': 'CCDS',
      'gender': 'Male',
      'cgpa': 4.02
    },
    ........
  ],
  "G-2": [
    .....
  ]
}
```






In [ ]:
tutorial_grp = {}

for student in student_list:
    tutorial = student["tutorial"]
    if tutorial not in tutorial_grp:
        tutorial_grp[tutorial] = []
    tutorial_grp[tutorial].append(student)

print(tutorial_grp)

# Create Helpers

To help ease the duplication of codes used in the algorithm, a set of helper functions are created.  
These functions handle team attribute counting, limit checking, and concentration evaluation, making the main algorithm cleaner and easier to maintain.

### Overview

- **`max_allowed_per_attribute(team_size)`**  
  Determines the maximum number of members allowed per attribute (e.g., gender) within a team.  
  Since the team size can be odd, the formula `(team_size + 1) // 2` ensures a fair distribution.  
  For example, for a team size of 5, the best ratio would be **3:2**.

- **`counts_in_team(team, key)`**  
  Builds a dictionary that accumulates the count of each attribute value (e.g., gender or nationality) within a team.  
  This helps in tracking how balanced the team composition is.

- **`violates_cap_after_add(team, student, key, team_size)`**  
  Checks whether adding a given student would cause the team to exceed the maximum allowed number for that attribute.  
  Returns `True` if the cap would be violated, otherwise `False`.

- **`concentration_after_add(team, student, key)`**  
  Calculates the projected count of an attribute after adding a student to the team.  
  Useful for evaluating how concentrated a particular attribute (e.g., same gender or same major) would become after the addition.




In [ ]:
def max_allowed_per_attribute(team_size):
    return (team_size + 1) // 2

def counts_in_team(team, key):
    counts_dict = {}
    for student in team:
        attribute_value = student[key]
        counts_dict[attribute_value] = counts_dict.get(attribute_value, 0) + 1
    return counts_dict

def violates_cap_after_add(team, student, key, team_size):
    max_allowed = max_allowed_per_attribute(team_size)
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1 > max_allowed

def concentration_after_add(team, student, key):
    current_counts = counts_in_team(team, key)
    attribute_value = student[key]
    return current_counts.get(attribute_value, 0) + 1

## Placement Scoring

`placement_score()` calculates how suitable it is to add a student to a team.  
Lower scores mean better fits.

### **Hard Constraints**
- If adding a student would exceed the team’s allowed gender or school limit → return a large penalty (`10,000`).

### **Soft Diversity Penalties**
- Add small costs for over-concentration:  
```
+5 × school_count
+3 × gender_count
```
####Academic balance:
- Penalize how far the team’s new mean t-score would drift from the tutorial’s mean t-score.

####Fill bias:
- Add a small penalty (`+1 × team_size`) to prefer less-filled teams.

####Result:
- The team with the lowest total score represents the most balanced choice for that student



In [ ]:
def placement_score(team, student, tutorial_mean_t, team_size):
    HARD = 10_000  # block if it creates a cap violation
    SOFT_SCHOOL = 5  # gentle penalty for concentration
    SOFT_GENDER = 3
    SOFT_FILL = 1  # slightly prefer smaller teams now

    # hard caps: do not place if it would exceed allowed count
    if violates_cap_after_add(team, student, "school", team_size):
        return HARD
    if violates_cap_after_add(team, student, "gender", team_size):
        return HARD

    # soft penalties: encourage spreading out attributes
    school_conc = concentration_after_add(team, student, "school")
    gender_conc = concentration_after_add(team, student, "gender")

    # CGPA balance (via t_score): keep team avg close to tutorial mean
    new_sum = sum(m["t_score"] for m in team) + student["t_score"]
    new_size = len(team) + 1
    cgpa_penalty = abs(new_sum / new_size - tutorial_mean_t)

    # fill penalty: slightly prefer teams with fewer members
    fill_penalty = len(team) * SOFT_FILL

    return (
        (SOFT_SCHOOL * school_conc)
        + (SOFT_GENDER * gender_conc)
        + cgpa_penalty
        + fill_penalty
    )

# Build Teams

The `build_teams()` function automatically forms balanced and diverse student teams within a tutorial group.  
It mixes students of different performance levels and backgrounds while keeping each team’s overall ability close to the class average.

### Steps Overview

1. **Sort by T-Score**  
   Students are first sorted by their `t_score` (academic performance + minority in schools/gender in tutorial group) in ascending order.

2. **Create "alternating high–low" Order**  
   To mix strong and weak students, the algorithm alternates between the highest and lowest scores, producing an order like:  
   `[highest, lowest, 2nd highest, 2nd lowest, ...]`  
   This ensures early placement of both top and bottom performers.

3. **Initialize Teams**  
   Calculates how many teams are needed (based on `team_size`) and creates empty team lists.

4. **Greedy Placement**  
   Iterates through students in the alternating high–low order:
   - For each student, tests every available team using `placement_score()`.  
   - Chooses the team with the lowest (best) placement score.  
   - If all teams are full, places the student in the smallest team as a fallback.

5. **Label & Finalize Teams**  
   After forming all teams:
   - Each team gets a unique ID (`tutorial_id-T1`, `tutorial_id-T2`, etc.).  
   - Computes each team’s average `t_score` and attaches it to members for reference.

### Notes
- `placement_score()` controls how good each placement is by considering:
  - Diversity (school, gender)
  - Academic balance (`t_score` closeness to mean)
  - Team fill preference

In [ ]:
def build_teams(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):
    # sort students by T score (ascending)
    students_sorted = sorted(students, key=lambda x: x["t_score"])

    # alternating high–low order: [highest, lowest, 2nd highest, 2nd lowest, ...]
    ahlo = []
    L = 0
    R = len(students_sorted) - 1
    while L <= R:
        ahlo.append(students_sorted[R]); R -= 1
        if L <= R:
            ahlo.append(students_sorted[L]); L += 1

    # correct number of teams (ceil division)
    num_of_teams = (len(students) + team_size - 1) // team_size
    teams = [[] for _ in range(num_of_teams)]

    # get the mean of a tutorial class
    tutorial_mean_t = sum(s["t_score"] for s in students) / len(students)

    # greedy placement
    # evaluate all available teams and calculate the placement scores to find the best fit
    for student in ahlo:
        best_team_index = None
        best_placement_score = float("inf")

        # enumerate to return both index and team
        for team_index, current_team in enumerate(teams):
            if len(current_team) >= team_size:
                continue
            score = placement_score(current_team, student, tutorial_mean_t, team_size)
            if score < best_placement_score:
                best_placement_score = score
                best_team_index = team_index

        if best_team_index is None:
            # safety fallback, pick smallest team
            best_team_index = min(range(len(teams)), key=lambda k: len(teams[k]))
        teams[best_team_index].append(student)

    # drop any accidental empties (extra safety)
    teams = [t for t in teams if t]

    # label teams and provide average tscore for each teams
    labeled = []
    for i, team in enumerate(teams, 1):
        team_id = f"{tutorial_id}-T{i}"
        team_mean = sum(x["t_score"] for x in team) / len(team)
        for x in team:
            x["team_id"] = team_id
            x["team_tscore"] = round(team_mean, 3)
        labeled.append(team)

    return labeled


## **Repair Teams**

The `repair_teams()` function performs a **targeted, iterative rebalancing** of teams after the initial greedy assignment.  
It repeatedly identifies the teams with the largest positive and negative deviations from the tutorial mean `t_score` and performs **cap-safe 1-for-1 swaps** between them to minimize imbalance.

### **How It Works**

1. **Identify Outliers**  
   - Compute each team’s mean `t_score` and its deviation from the tutorial mean.  
   - Group teams into “high” (above `+threshold`) and “low” (below `−threshold`) sets.

2. **Search for Best Swap**  
   - For every high–low team pair, test all possible student swaps.  
   - Skip swaps that would exceed **school** or **gender caps**.  
   - For each valid swap, calculate how much it reduces the **total deviation** from the tutorial mean.  
   - Keep the swap with the **largest improvement**.

3. **Commit and Repeat**  
   - Apply the best swap found in the current iteration.  
   - Update `team_tscore` for the affected teams.  
   - Repeat the process until no improving swap exists or the iteration limit (`2 × number_of_teams`) is reached.

### **Summary**

- Performs **repeated 1-for-1 swaps** to reduce imbalance  
- Preserves **team size**, **school caps**, and **gender caps**  
- Stops automatically when no better swaps remain or a safety limit is reached  
- Gradually pulls team averages toward the tutorial mean without reassigning everyone


In [ ]:
def repair_teams(teams, team_size, tutorial_mean_t, threshold=0.1):

    def team_mean(team):
        return sum(s["t_score"] for s in team) / len(team)

    def deviation(team):
        return team_mean(team) - tutorial_mean_t

    cap = max_allowed_per_attribute(team_size)

    # check if 1-for-1 swap preserves school/gender caps
    def cap_ok_after_swap(team_a, team_b, student_a, student_b):

        # max allowed counts based on each team's current size
        cap_a = max_allowed_per_attribute(len(team_a))
        cap_b = max_allowed_per_attribute(len(team_b))

        for attribute in ("school", "gender"):
            a_counts = counts_in_team(team_a, attribute)
            b_counts = counts_in_team(team_b, attribute)

            # simulate removing the chosen students from their current teams
            a_counts[student_a[attribute]] = a_counts.get(student_a[attribute], 0) - 1
            b_counts[student_b[attribute]] = b_counts.get(student_b[attribute], 0) - 1

            # simulate adding the incoming students after the swap
            a_counts[student_b[attribute]] = a_counts.get(student_b[attribute], 0) + 1
            b_counts[student_a[attribute]] = b_counts.get(student_a[attribute], 0) + 1

            # check if variations exceeds cap
            # i.e 3 is the cap for a team of 5 people, so cap_a = 3
            # v in a_counts.values() -> gets the current variations in numbers, [3,4]
            # now we use v to check if it goes over the cap
            # return false if it goes over
            if any(v > cap_a for v in a_counts.values()):
              return False
            if any(v > cap_b for v in b_counts.values()):
              return False
        return True

    # prevents unlimited iteration
    max_iterations = len(teams) * 2
    iteration = 0

    # repair teams whose mean deviates beyond +-threshold
    while iteration < max_iterations:
        iteration += 1

        # identify teams too high/low to target threshold
        highs = [t for t in teams if deviation(t) > threshold]
        lows = [t for t in teams if deviation(t) < -threshold]

        if not highs or not lows:
            break

        best_pair = None
        best_cost = None

        # try every combination of one high-deviation team and one low-deviation team
        for hi in highs:
            hi_mean = team_mean(hi)
            hi_size = len(hi)
            for lo in lows:
                lo_mean = team_mean(lo)
                lo_size = len(lo)

                # how far both teams currently are from the tutorial target mean
                base_cost = abs(hi_mean - tutorial_mean_t) + abs(lo_mean - tutorial_mean_t)

                # try swapping every possible pair of students (one from hi, one from lo)
                for i, s_hi in enumerate(hi):
                    for j, s_lo in enumerate(lo):

                        # check for violations on school/gender caps
                        if not cap_ok_after_swap(hi, lo, s_hi, s_lo):
                            continue

                        # calculate new high and low means
                        new_hi_mean = hi_mean + (s_lo["t_score"] - s_hi["t_score"]) / hi_size
                        new_lo_mean = lo_mean + (s_hi["t_score"] - s_lo["t_score"]) / lo_size
                        new_cost = abs(new_hi_mean - tutorial_mean_t) + abs(new_lo_mean - tutorial_mean_t)

                        # only take swaps that make each high and low teams deviate closer to the tutorial mean
                        if new_cost < base_cost:
                            if best_cost is None or new_cost < best_cost:
                                best_cost = new_cost
                                # save into best pair for swapping
                                best_pair = (hi, lo, i, j, new_hi_mean, new_lo_mean, new_cost)

        # stop if no improvememts from changes
        if best_pair is None:
            break

        # extract data from best_pair and commit the swap
        hi, lo, i, j, new_hi_mean, new_lo_mean, _ = best_pair

        # Commit the swap
        hi[i], lo[j] = lo[j], hi[i]

        # Update stored team_tscore for affected teams
        new_hi = team_mean(hi)
        new_lo = team_mean(lo)
        for m in hi:
            m["team_tscore"] = round(new_hi, 3)
        for m in lo:
            m["team_tscore"] = round(new_lo, 3)

    return teams

## Create Groups by T-Score

The `create_groups_by_tscore()` function builds diverse, balanced teams for a tutorial.  
It combines **academic normalization**, **diversity bonuses**, **placement scoring**, and a **repair phase** to optimize team composition.

### Workflow Overview

1. **Normalize CGPA → Compute T-Scores**  
   - Each student’s CGPA is scaled between 0 and 1 based on the tutorial’s min/max range.  
   - Diversity bonuses are added:  
     - **+0.2** if from the least represented gender  
     - **Up to +0.1**, scaled according to how underrepresented the student’s school is compared to the average number of students per school in the tutorial group
        - This provides fair consideration for smaller schools without overpowering the CGPA component.
   - The resulting `t_score` reflects both ability and diversity value.

2. **Compute Tutorial Mean**  
   Calculates the average `t_score` across all students to use as a balancing reference.

3. **Initial Team Formation**  
   Calls `build_teams()` to assign students into teams using the diversity-aware placement algorithm.  
   Teams are built to respect size and attribute caps.

4. **Repair Phase**  
   Calls `repair_teams()` to rebalance teams, refining mean balance and reducing cap violations.

5. **Summary Output**  
   Prints overall and per-team stats:
   - Tutorial mean T-score  
   - Average team deviation from the mean  
   - Each team’s ID, mean T-score, and deviation

### Key Notes
- `t_score` = normalized CGPA + diversity bonuses (bonus given to minority)
- The algorithm ensures teams are **diverse**, **balanced**, and **close to the overall mean performance**.
- Ideally, the target should not exceed > 0.1 for each tutorial group against the tutorial means

In [ ]:
def create_groups_by_tscore(tutorial_id, students, team_size=DEFAULT_TEAM_SIZE):

    # min/max CGPA for normalization
    cgpas = [student["cgpa"] for student in students]

    min_cgpa = min(cgpas)
    max_cgpa = max(cgpas)
    cgpa_range = max_cgpa - min_cgpa

    # counts in tutorial
    gender_counts = {}
    school_counts = {}
    for student in students:
        gender_counts[student["gender"]] = gender_counts.get(student["gender"], 0) + 1
        school_counts[student["school"]] = school_counts.get(student["school"], 0) + 1

    min_gender_count = min(gender_counts.values()) if gender_counts else 0
    min_school_count = min(school_counts.values()) if school_counts else 0

    num_schools = max(1, len(school_counts))
    avg_per_school = len(students) / num_schools  # dynamic baseline from school_counts

    # dynamic school bonus func
    def school_bonus(school):
      if num_schools == 1:
          return 0.0
      c = school_counts[school]
      rarity = max(0.0, min(1.0, (avg_per_school - c) / avg_per_school))
      return 0.1 * rarity

    # compute t-scores for each students
    total_tscore = 0
    for student in students:
        normalized_cgpa = (student["cgpa"] - min_cgpa) / cgpa_range
        gender_diversity_bonus = (
            0.2 if gender_counts[student["gender"]] == min_gender_count else 0.0
        )
        school_diversity_bonus = (
            school_bonus(student["school"]) if school_counts[student["school"]] == min_school_count else 0.0
        )
        t_score = normalized_cgpa + gender_diversity_bonus + school_diversity_bonus

        student["t_score"] = round(t_score, 3)
        total_tscore += student["t_score"]

    # tutorial mean T-score
    mean_tscore = total_tscore / len(students)

    # build initial teams with placement
    teams = build_teams(tutorial_id, students, team_size)

    # repair teams to polish t_score
    teams = repair_teams(teams, team_size, mean_tscore)

    # summary printout
    team_mean_diffs = [
        abs(sum(student["t_score"] for student in t) / len(t) - mean_tscore)
        for t in teams
    ]
    avg_team_diff = sum(team_mean_diffs) / len(team_mean_diffs)

    print(f"\n=== {tutorial_id} summary ===")
    print(f"Tutorial mean T-score: {mean_tscore:.3f}")
    print(f"Actual average team diff from tutorial mean: {avg_team_diff:.4f}")

    for team in teams:
        team_mean = sum(s["t_score"] for s in team) / len(team)
        diff = abs(team_mean - mean_tscore)
        print(
            f"{tutorial_id} | {team[0]['team_id']} | Team mean T={team_mean:.3f} | Diff={diff:.3f}"
        )

    return teams

In [ ]:
# run algorithm
sorted_tutorial_grp = {}

for tutorial_id, students in tutorial_grp.items():
    teams = create_groups_by_tscore(tutorial_id, students, TEAM_SIZE)
    sorted_tutorial_grp[tutorial_id] = teams

In [ ]:
# export to csv
output_data = []

for tutorial_id, teams in sorted_tutorial_grp.items():
    for team in teams:
        for student in team:
            row = {
                "Tutorial Group": student["tutorial"],
                "Student ID": student["student_id"],
                "School": student["school"],
                "Name": student["name"],
                "Gender": student["gender"],
                "CGPA": student["cgpa"],
                "Team Assigned": student.get("team_id", f"{tutorial_id}-Unknown"),
            }
            output_data.append(row)

output_file = "./output.csv"
fieldnames = [
    "Tutorial Group",
    "Student ID",
    "School",
    "Name",
    "Gender",
    "CGPA",
    "Team Assigned",
]

with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_data)

print(f"\nTeam assignments exported to {output_file}")
print(f"Total students processed: {len(output_data)}")
print(
    f"Total teams created: {sum(len(teams) for teams in sorted_tutorial_grp.values())}"
)

In [ ]:
import matplotlib.pyplot as plt

# helpers
def _find_field_name(sample_row, candidates):
    lower_map = {k.lower(): k for k in sample_row.keys()}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _teams_from_rows(rows):
    teams = {}
    for r in rows:
        tid = (
            r.get("Team Assigned")
            or r.get("team assigned")
            or r.get("team")
            or r.get("Team")
        )
        if tid is None:
            continue
        teams.setdefault(str(tid), []).append(r)
    return teams  # dict: {team_id: [rows...]}


def _to_float(val):
    if val is None:
        return None
    s = str(val).strip().replace(",", "")
    if not s:
        return None
    try:
        return float(s)
    except:
        return None


def _gender_code(val):
    """Return 'M' or 'F' if clearly identifiable, else None."""
    if val is None:
        return None
    s = str(val).strip().lower()
    if not s:
        return None
    if s[0] == "m":
        return "M"
    if s[0] == "f":
        return "F"
    return None


# gender-mix distribution chart
def plot_gender_mix_distribution(team_data_rows, team_size_hint=None):
    if not team_data_rows:
        print("No data to plot (gender mix).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (gender mix).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    gender_col = _find_field_name(any_row, ["Gender"])
    if gender_col is None:
        print("Couldn't find a 'Gender' column.")
        return

    from collections import Counter, defaultdict

    comp_counts_by_size = defaultdict(Counter)  # {size: Counter("xM+yF" -> n)}
    size_counts = Counter()

    for members in teams_dict.values():
        m = f = 0
        for r in members:
            g = _gender_code(r.get(gender_col))
            if g == "M":
                m += 1
            elif g == "F":
                f += 1
        size = len(members)
        if team_size_hint is not None:
            # lock bins to a fixed size if desired
            if m + f != team_size_hint:
                # if blanks/others exist, force F from hint
                f = max(0, team_size_hint - m)
            size = team_size_hint
        else:
            size = m + f  # only count rows recognized as M/F

        label = f"{m}M+{f}F"
        comp_counts_by_size[size][label] += 1
        size_counts[size] += 1

    if not size_counts:
        print("No recognizable M/F data to plot.")
        return

    primary_size = max(size_counts.items(), key=lambda kv: kv[1])[0]

    def _plot_size(size, counts):
        labels = [f"{m}M+{size-m}F" for m in range(size, -1, -1)]
        values = [counts.get(lbl, 0) for lbl in labels]

        fig = plt.figure(figsize=(8, 4.5))
        ax = fig.gca()
        ax.bar(range(len(labels)), values)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_xlabel("Team gender composition")
        ax.set_ylabel("Number of teams")
        ax.set_title(f"Gender Mix Distribution (Team Size {size})")
        for x, v in enumerate(values):
            if v > 0:
                ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
        plt.tight_layout()
        plt.show()

    # plot the most common team size first
    _plot_size(primary_size, comp_counts_by_size[primary_size])

    # plot other sizes if present
    for size, counts in comp_counts_by_size.items():
        if size == primary_size:
            continue
        _plot_size(size, counts)


# teams by number of distinct schools chart
def plot_school_count_distribution(team_data_rows):
    if not team_data_rows:
        print("No data to plot (schools).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (schools).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    school_col = _find_field_name(
        any_row, ["School"]
    )
    if school_col is None:
        print("Couldn't find a 'School' column.")
        return

    from collections import Counter

    counter = Counter()
    max_sch = 0

    for members in teams_dict.values():
        schools = []
        for r in members:
            val = r.get(school_col)
            val = "" if val is None else str(val).strip()
            if val:
                schools.append(val)
        distinct = len(set(schools))
        counter[distinct] += 1
        if distinct > max_sch:
            max_sch = distinct

    xs = list(range(1, max_sch + 1)) if max_sch > 0 else [0]
    ys = [counter.get(x, 0) for x in xs]

    fig = plt.figure(figsize=(8, 4.5))
    ax = fig.gca()
    ax.bar(range(len(xs)), ys)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([str(x) for x in xs])
    ax.set_xlabel("Number of distinct schools within a team")
    ax.set_ylabel("Number of teams")
    ax.set_title("School Diversity: Teams by Count of Distinct Schools")
    for x, v in enumerate(ys):
        if v > 0:
            ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()


# average CGPA per team with overall line chart
def plot_average_cgpa_per_team(team_data_rows, show_hist=True):
    if not team_data_rows:
        print("No data to plot (CGPA).")
        return

    teams_dict = _teams_from_rows(team_data_rows)
    if not teams_dict:
        print("No teams found (CGPA).")
        return

    any_row = next(iter(teams_dict.values()))[0]
    cgpa_col = _find_field_name(any_row, ["CGPA"])
    if cgpa_col is None:
        print("Couldn't find a 'CGPA' column.")
        return

    team_ids, team_avgs, all_cgpas = [], [], []

    for tid, members in sorted(teams_dict.items(), key=lambda kv: kv[0]):
        vals = []
        for r in members:
            v = _to_float(r.get(cgpa_col))
            if v is not None:
                vals.append(v)
                all_cgpas.append(v)
        if vals:
            avg = sum(vals) / len(vals)
            team_ids.append(str(tid))
            team_avgs.append(avg)

    overall_avg = (sum(all_cgpas) / len(all_cgpas)) if all_cgpas else 0.0

    fig2 = plt.figure(figsize=(7.5, 4.5))
    ax2 = fig2.gca()
    bins = max(5, min(12, len(team_avgs)))
    ax2.hist(team_avgs, bins=bins)
    ax2.set_xlabel("Team average CGPA")
    ax2.set_ylabel("Number of teams")
    ax2.set_title("Distribution of Team Average CGPA")
    ax2.axvline(overall_avg, color="red", linewidth=2)
    ymax = ax2.get_ylim()[1]
    ax2.text(
        overall_avg,
        ymax * 0.95,
        f"{overall_avg:.2f}",
        ha="left",
        va="top",
        fontsize=9,
        color="red",
    )
    plt.tight_layout()
    plt.show()


# (adjust team_size_hint if teams are fixed size, e.g., 5)
plot_gender_mix_distribution(output_data, team_size_hint=TEAM_SIZE)
plot_school_count_distribution(output_data)
plot_average_cgpa_per_team(output_data, show_hist=True)

#Documentation & Report

###Group members:
1. Goh Kai Xuan, Bryan (bgoh031@e.ntu.edu.sg)
2. Edwin Tan Qi Lun  (ETAN144@e.ntu.edu.sg)
3. Jaden Loh Jing Han  (jloh065@e.ntu.edu.sg)
4. Ivan Tey Yi Kai   (IVAN0070@e.ntu.edu.sg)
5. Hoo Wen Xuan (WHOO001@e.ntu.edu.sg)

### Problem Analysis

The task involves forming balanced and diverse teams from a dataset of **6,000 students** divided into **120 tutorial groups**, with each group containing **50 students**. Each team must consist of **five members**, and all members must belong to the same tutorial group.

The main objective is to create **balanced** teams by considering three key factors,  **school affiliation, gender, and CGPA**.  
No team should have a majority from the same school or the same gender, and teams should also maintain a balanced range of academic performance based on CGPA.

---

###  Problem Decomposition Process

> To tackle the challenge of forming balanced and diverse teams, the main problem was broken down into the following smaller tasks:



1. **Read the CSV file containing student information**  
   - Load all student data including Tutorial Group, Student ID, Name, School, Gender, and CGPA.  
   - Ensure data is clean and correctly formatted for further processing.  

2. **Organize students by their tutorial group**  
   - Group all students into their respective tutorial groups to ensure teams are only formed within the same tutorial since teams are not mixed across different tutorials.  

3. **Determine the ideal team combination**  
   - Maintain a **x:x (even), x:x+1 (odd) gender ratio** per team to promote gender diversity.  
   - Ensure team members come from **different schools**
   - Keep the **average team CGPA balanced** across all teams within a tutorial group to avoid teams dominated by only students with high CPGAs or low CGPAs.  
   - This is the ideal team that the algorithm will aim to achieve.  

4. **Build the teams following the ideal combination**  
   - Sort students based on a **t_score**, which considers their relative CGPA, gender, and school representation.  
   - Use a **placement score** to determine which team each student should be placed such that the team is closer to ideal combination
   - Assign team IDs and calculate each team’s average t_score.  
   - teams are built as balanced as possible without violating gender or school limits.  


5. **Plot graphs to visualize how many teams are well-balanced and how many are not**
   - After forming the teams, we generate charts to evaluate diversity and fairness.

   - Visualizations include gender distribution, school representation, and average CGPA per team.

   - These graphs help identify how many teams are well-balanced and how many teams may need improvement.

6. **Export the final team back to a CSV file**
   - Save all team assignments, including student information and team IDs, into a CSV file

#  Pseudocode

---

##  1. Read the CSV File and Sort Students by Tutorial Group

> Read the student data and organize them by tutorial group.
```
Read the CSV file  
For each row in the file:  
    Create a dictionary for each student containing:  
    Tutorial Group, Student ID, Name, School, Gender, and CGPA  
    Add the student dictionary to student_list  

Group all students in student_list by their tutorial group
```

---

##  2. Calculate the t_score and placement_score

> Compute each student’s t_score to measure performance and diversity contribution,  
> and calculate placement_score to decide which team they fit best in.
```
For each student in the tutorial group:
    Calculate their relative CGPA within the tutorial group
    Add +0.2 bonus if the student’s gender is least represented
    Add a dynamic school bonus up to +0.1 based on how underrepresented the student’s school is
        (rarer schools receive a higher bonus, while common schools receive little or no bonus)
    Calculate t_score = relative_CGPA + gender_bonus + school_bonus
    Store t_score for each student in the student_list

For each tutorial group:
    Sort students by t_score in alternating high–low order:
        [highest, lowest, 2nd highest, 2nd lowest, and so on]
    Calculate number_of_teams = total_students / team_size
    Create empty team lists
    Compute mean_tscore for the tutorial group

    For each student:
        Calculate placement_score for each available team as:
            (difference between team’s new average_tscore if added and tutorial mean_tscore)
            + penalty if adding the student exceeds gender or school diversity limits
        Place the student into the team with the lowest placement_score

```

---

##  3. Building Teams

> Distribute students into teams using the placement score to ensure fairness and diversity.
```
For each student in the alternating high-low order list:  
    Find the team with the lowest placement_score that is not full  
    If no such team exists:  
        Place student in the team with the smallest team size  

For each team:  
    Assign a team_id  
    Calculate team average t_score  
    Store team_id and team_tscore for each student  
```

---

##  4. Improving Teams

> Balance the teams by swapping members between teams that deviate too much  
> from the tutorial group’s average t_score.
```
For students in teams with threshold > +-0.1 difference from tutorial mean t_score:  
    If swapping them makes both team averages closer to the tutorial mean t_score  
    without violating gender or school limits:  
        Swap the students  
    Else:  
        Continue  

Update the new teams  
```

---

##  5. Export the Team Details to CSV

> Save the finalized teams back into a CSV file for record keeping.
```
For each team:  
    For each student:  
        Create a row with student info + team_id  
Add all rows to CSV file  
```

---

##  6. Generate Charts for Analysis

> Visualize the team diversity and balance using charts.
```

For each team:  
    Count number of male and female students  
    Plot a bar chart showing male vs female counts per team  

For each team:  
    Count number of distinct schools  
    Plot a bar chart showing how many teams have 1 school, 2 schools, etc.  

For each team:  
    Calculate average CGPA  
Plot a histogram of team average CGPAs  
Draw a line showing the overall average CGPA  
```
---

##Challenges and issues faced
Placement Score Calculation
- Balancing the penalty weights on multiple factors like CGPA, gender, and school can be difficult; if set too high or too low, teams may become unbalanced.

- Worst-case conditions have to be considered, such as when a student cannot be placed because both gender and school limits are reached, or when two or more teams have the same placement score for a student, making tie-breaking necessary. We handle this by assigning the student to the smallest team.

As most of us are new to python, creating the graph was slightly harder than usual as we have to read the library's documentation to understand how it works.



---

### Conclusion
In conclusion, the project successfully formed balanced and diverse teams based on school, gender, and CGPA. By applying a structured algorithm and fairness rules, the teams were built while ensuring diversity within each tutorial group as much as possible. Despite some limitations due to uneven distributions, the approach achieved fair and reasonable results across all tutorial groups. Despite all the difficulties faced, we gained a much better understanding of Python, data handling, and visualization through this project. It was a steep learning curve, but it helped us improve our coding and teamwork skills a lot.




